# V1.4










### 分析中notebook

|名称|notebook名称|notebook URL|
|----|----|----|
|メイン|kaggriculture-movements-top-xx|**https://www.kaggle.com/code/nagatakengo/kaggriculture-movements-top-xx**<br><br><br>|
|社長と従業員の行動範囲|range_of_action|**https://www.kaggle.com/code/nagatakengo/range-of-action**<br><br><br>|
|コストと物価|costs|**https://www.kaggle.com/code/nagatakengo/costs**<br><br><br>|

In [1]:
!pip install -qq kaggle-environments

# 提出関数

In [4]:
%%writefile main.py
import random


def get_market_price(obs, product):
    """現在価格を取得する。"""
    return obs["market"]["prices"][product]


def should_sell_melon(melon_price, day, step):
    """MELONをSELLするかHOLDするか判断する。"""

    # 終盤は価格に関係なく売却
    if step >= 710:
        return True

    # 基準価格以上なら売却
    if melon_price >= 250:
        return True

    return False


def get_harvest_age(crop_name):
    """現在の戦略で使う収穫開始日を返す。"""
    if crop_name == "MELON":
        return 10

    return 2


def get_tile_action(tile, day):
    """足元のタイルで今すぐ行う作業を返す。"""
    if not isinstance(tile, dict):
        return None

    if tile.get("kind") == "WEED":
        return ["DIG"]

    if tile.get("kind") != "PLANT":
        return None

    crop_name = tile.get("crop", "WHEAT")
    crop_age = day - tile.get("planted_day", day)
    harvest_age = get_harvest_age(crop_name)

    if crop_age >= harvest_age:
        return ["HARVEST"]

    if not tile.get("watered_today", True):
        return ["WATER"]

    return None


def step_toward(fx, fy, tx, ty, tiles):
    """
    目的地に近づく方向へ1マス移動する。
    目的地へ直接進めない場合は、移動可能な方向を選ぶ。
    """
    max_y = len(tiles) - 1
    max_x = len(tiles[0]) - 1

    candidates = []

    if fx < tx and fx < max_x and tiles[fy][fx + 1] != "LOCKED":
        candidates.append("EAST")
    elif fx > tx and fx > 0 and tiles[fy][fx - 1] != "LOCKED":
        candidates.append("WEST")

    if fy < ty and fy < max_y and tiles[fy + 1][fx] != "LOCKED":
        candidates.append("SOUTH")
    elif fy > ty and fy > 0 and tiles[fy - 1][fx] != "LOCKED":
        candidates.append("NORTH")

    if candidates:
        return random.choice(candidates)

    valid_dirs = []

    if fy > 0 and tiles[fy - 1][fx] != "LOCKED":
        valid_dirs.append("NORTH")

    if fy < max_y and tiles[fy + 1][fx] != "LOCKED":
        valid_dirs.append("SOUTH")

    if fx < max_x and tiles[fy][fx + 1] != "LOCKED":
        valid_dirs.append("EAST")

    if fx > 0 and tiles[fy][fx - 1] != "LOCKED":
        valid_dirs.append("WEST")

    if valid_dirs:
        return random.choice(valid_dirs)

    return "PASS"


def find_target_tile(
    tiles,
    fx,
    fy,
    has_seeds,
    day,
    excluded_coords=None,
):
    """
    現在位置から各作業候補を評価し、
    最もスコアの高いタイルの座標を返す。
    """
    if excluded_coords is None:
        excluded_coords = set()

    best_target = None
    best_score = -9999

    for y in range(len(tiles)):
        for x in range(len(tiles[0])):

            if (x, y) in excluded_coords:
                continue

            tile = tiles[y][x]

            if tile == "LOCKED":
                continue

            base_score = None

            # 雑草
            if isinstance(tile, dict) and tile.get("kind") == "WEED":
                if day >= 27:
                    continue

                base_score = 0

            # 植物
            elif isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop_name = tile.get("crop", "WHEAT")
                crop_age = day - tile.get("planted_day", day)
                harvest_age = get_harvest_age(crop_name)

                if crop_age >= harvest_age:
                    if crop_name == "MELON":
                        base_score = 250
                    else:
                        base_score = 25

                elif not tile.get("watered_today", True):
                    base_score = 50

            # 空き地
            elif tile is None and has_seeds:
                if day >= 27:
                    continue

                base_score = 50

            if base_score is None:
                continue

            movement_cost = abs(x - fx) + abs(y - fy)
            task_score = base_score - movement_cost

            if task_score > best_score:
                best_score = task_score
                best_target = (x, y)

    return best_target


def build_market_actions(
    me,
    seeds,
    shed,
    day,
    step,
    melon_price,
):
    """現在の市場売買・雇用・土地購入ルールから注文一覧を作る。"""

    market = []

    money = me.get("money", 0)
    current_hands = me.get("hands", [])
    unlocked_quads = me.get(
        "unlocked_quadrants",
        ["NW"],
    )

    wheat_seeds = seeds.get("WHEAT", 0)
    melon_seeds = seeds.get("MELON", 0)

    wheat_in_shed = shed.get("WHEAT", 0)
    melon_in_shed = shed.get("MELON", 0)

    # -------------------------
    # 種を購入
    # -------------------------

    if money >= 500:
        if melon_seeds == 0:
            market.append(
                ["BUY_SEED", "MELON", 1]
            )

    elif wheat_seeds == 0 and money >= 10:
        market.append(
            ["BUY_SEED", "WHEAT", 1]
        )

    # -------------------------
    # WHEAT売却
    # -------------------------

    if wheat_in_shed > 0:
        market.append(
            ["SELL", "WHEAT", wheat_in_shed]
        )

    # -------------------------
    # MELON売却
    # -------------------------

    if melon_in_shed > 0:
        if should_sell_melon(
            melon_price,
            day,
            step,
        ):
            market.append(
                ["SELL", "MELON", melon_in_shed]
            )

    # -------------------------
    # 雇用
    # -------------------------

    total_people = 1 + len(current_hands)

    if total_people < 4 and money >= 200:
        market.append(
            ["HIRE"]
        )

    # -------------------------
    # 土地購入
    # -------------------------

    if len(unlocked_quads) < 2 and money >= 1000:
        market.insert(
            0,
            ["BUY_LAND"],
        )

    return market


def agent(obs, config):

    # ========================================
    # 状態取得
    # ========================================

    player = obs["player"]

    me = obs["farms"][player]
    private = obs["private"]

    tiles = me["tiles"]

    fx, fy = me["farmer"]
    farmer_tile = tiles[fy][fx]

    seeds = private.get("seeds", {})
    shed = private.get("shed", {})

    day = obs.get("day", 0)
    step = obs.get("step", 0)

    current_hands = me.get("hands", [])

    # 現在のMELON価格を取得
    melon_price = get_market_price(
        obs,
        "MELON",
    )

    wheat_seeds = seeds.get("WHEAT", 0)
    melon_seeds = seeds.get("MELON", 0)

    has_seeds = (
        wheat_seeds > 0
        or melon_seeds > 0
    )

    # ========================================
    # 市場
    # ========================================

    market = build_market_actions(
        me,
        seeds,
        shed,
        day,
        step,
        melon_price,
    )

    # ========================================
    # メイン農家
    # ========================================

    farmer_action = None

    if farmer_tile is None:

        if melon_seeds > 0:
            farmer_action = [
                "PLANT",
                "MELON",
            ]

        elif wheat_seeds > 0:
            farmer_action = [
                "PLANT",
                "WHEAT",
            ]

    else:
        farmer_action = get_tile_action(
            farmer_tile,
            day,
        )

    # ========================================
    # メイン農家の移動
    # ========================================

    claimed_targets = set()

    if farmer_action is None:

        target = find_target_tile(
            tiles,
            fx,
            fy,
            has_seeds,
            day,
            claimed_targets,
        )

        if target is not None:

            claimed_targets.add(target)

            move_dir = step_toward(
                fx,
                fy,
                target[0],
                target[1],
                tiles,
            )

            farmer_action = [move_dir]

        else:
            farmer_action = ["PASS"]

    else:
        claimed_targets.add(
            (fx, fy)
        )

    # ========================================
    # 作業員
    # ========================================

    hands_actions = []

    for hand in current_hands:

        hx, hy = hand
        hand_tile = tiles[hy][hx]

        hand_action = get_tile_action(
            hand_tile,
            day,
        )

        if hand_action is None:

            target = find_target_tile(
                tiles,
                hx,
                hy,
                has_seeds,
                day,
                claimed_targets,
            )

            if target is not None:

                claimed_targets.add(target)

                move_dir = step_toward(
                    hx,
                    hy,
                    target[0],
                    target[1],
                    tiles,
                )

                hand_action = [move_dir]

            else:
                hand_action = ["PASS"]

        else:
            claimed_targets.add(
                (hx, hy)
            )

        hands_actions.append(
            hand_action
        )

    # ========================================
    # 出力
    # ========================================

    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market,
    }

Overwriting main.py


# テスト

In [3]:
from zoneinfo import ZoneInfo
import datetime
from kaggle_environments import make
from IPython.display import HTML



print(datetime.datetime.now(ZoneInfo("Asia/Tokyo")))


env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)

env.run(["main.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

"""
html_output = env.render(mode="html", width=500, height=500)
HTML(html_output)
"""

2026-08-16 23:43:26.764049+09:00
Player 0: reward=27717.0, status=DONE
Player 1: reward=0.0, status=DONE


'\nhtml_output = env.render(mode="html", width=500, height=500)\nHTML(html_output)\n'

# メモ

1. get_harvest_age()
   作物の収穫開始日を返す

2. get_tile_action()
   足元で DIG / HARVEST / WATER のどれをするか判断する

3. step_toward()
   目的地へ1マス移動する

4. find_target_tile()
   次に向かう作業場所を探す

5. build_market_actions()
   種購入 / 売却 / 雇用 / 土地購入の注文を作る

6. agent()
   状態を取得して、
   上記の機能を呼び出し、
   最終的な行動を返す